In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# Step 2: Load Dataset
# Use the local filename since the file is already in the environment
data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')

In [ ]:
# 2. Clean Column Names (Removes leading spaces like ' Label' -> 'Label')
data.columns = data.columns.str.strip()

In [ ]:
# 3. Data Preprocessing
# Replace Infinity values with NaN and drop them
data.replace([np.inf, -np.inf], np.nan, inplace=True)
data.dropna(inplace=True)

# Convert categorical Labels to numbers: BENIGN = 0, DDoS = 1
data['Label'] = data['Label'].map({'BENIGN': 0, 'DDoS': 1})

In [ ]:
# 4. Feature Selection
# We use numeric columns only. We drop 'Label' (target) and non-numeric metadata
X = data.select_dtypes(include=[np.number]).drop(columns=['Label'])
y = data['Label']

In [ ]:
# Step 5: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

In [ ]:
# 6. Create and Train Random Forest Model
# n_estimators=100 means 100 decision trees will work together
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

In [ ]:
# 7. Make Predictions
y_pred = rf_model.predict(X_test)

In [ ]:
# 8. Evaluation Metrics
print("\n--- Model Performance ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['BENIGN', 'DDoS']))

In [ ]:
# 9. Visualize the Results (Confusion Matrix)
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['BENIGN', 'DDoS'], yticklabels=['BENIGN', 'DDoS'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('DDoS Detection Confusion Matrix')
plt.show()

In [ ]:
# 10. Feature Importance (Bonus: See which network features mattered most)
importances = rf_model.feature_importances_
indices = np.argsort(importances)[-10:] # Top 10 features

plt.figure(figsize=(10, 6))
plt.title('Top 10 Important Network Features')
plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [X.columns[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

In [ ]:
# 1. Grab a 'Safe' example from the test set
safe_example = X_test.iloc[0:1]
prediction_1 = rf_model.predict(safe_example)
print(f"Result 1: {'DDoS' if prediction_1[0] == 1 else 'Safe'}")

# 2. Grab a 'DDoS' example from the test set
# We find where labels are 1 (DDoS)
ddos_index = y_test[y_test == 1].index[0]
ddos_example = X_test.loc[[ddos_index]]
prediction_2 = rf_model.predict(ddos_example)
print(f"Result 2: {'DDoS' if prediction_2[0] == 1 else 'Safe'}")

# 3. Create a manual 'Attack' scenario by modifying a row
# Let's take a safe row and drastically increase the 'Bwd Packet Length Std'
# and decrease 'Flow Duration' to simulate a flood.
test_input = safe_example.copy()
test_input['Bwd Packet Length Std'] = 500.0
test_input['Flow Duration'] = 10
prediction_3 = rf_model.predict(test_input)
print(f"Manual Test Result: {'DDoS' if prediction_3[0] == 1 else 'Safe'}")